In [2]:
import time
import numpy as np
import cv2
import tracemalloc
from pynq import Overlay


def cartoon_filter(
    
    bgr: np.ndarray,
    edge_thresh: int = 150,
    bright_factor: int = 320,   # fixed-point: 256 == 1.0
    quant_levels: int = 6,
    thicken_horizontal: bool = True,
) -> np.ndarray:
    H, W = bgr.shape[:2]

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    gray_blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    gx = cv2.Sobel(gray_blurred, cv2.CV_32F, 1, 0, ksize=3, borderType=cv2.BORDER_REPLICATE)
    gy = cv2.Sobel(gray_blurred, cv2.CV_32F, 0, 1, ksize=3, borderType=cv2.BORDER_REPLICATE)

    mag = cv2.magnitude(gx, gy)
    edge = mag > edge_thresh

    if thicken_horizontal:
        edge_u8 = edge.astype(np.uint8) * 255
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 1))
        thick_edge = cv2.dilate(edge_u8, kernel, iterations=1).astype(bool)
    else:
        thick_edge = edge

    blurred = cv2.blur(bgr, (3, 3), borderType=cv2.BORDER_REPLICATE)

    v = (blurred.astype(np.int32) * bright_factor) >> 8
    v = np.clip(v, 0, 255).astype(np.uint8)

    step = 256 // max(1, quant_levels)
    out = ((v // step) * step).astype(np.uint8)

    valid = np.zeros((H, W), dtype=bool)
    if H > 2 and W > 2:
        valid[2:, 2:] = True

    out[thick_edge & valid] = (0, 0, 0)

    border_mask = ~valid
    orig = (bgr.astype(np.int32) * bright_factor) >> 8
    orig = np.clip(orig, 0, 255).astype(np.uint8)
    orig = ((orig // step) * step).astype(np.uint8)
    out[border_mask] = orig[border_mask]

    return out

In [3]:
def main():
    EDGE_THRESH = 150
    BRIGHT_FACTOR = 320
    QUANT_LEVELS = 6
    THICKEN = True

    WARMUP_FRAMES = 20
    TEST_FRAMES = 200

    ENABLE_HDMI_OUT = False   # set False if you only want compute timing

    # ---- Import PYNQ and configure HDMI ----
    try:
        cartoon_filter_overlay = Overlay("cartoon_filter.bit")
    except Exception:
        print("ERROR: Could not load the overlay.")
        return

    try:
        hdmi_in = cartoon_filter_overlay.video.hdmi_in
    except Exception:
        print("ERROR: Could not access cartoon_filter_overlay.video.hdmi_in.")
        return

    hdmi_out = None
    if ENABLE_HDMI_OUT:
        try:
            hdmi_out = cartoon_filter_overlay.video.hdmi_out
        except Exception:
            print("WARNING: Could not access cartoon_filter_overlay.video.hdmi_out")
            hdmi_out = None

    print("Configuring HDMI_IN...")
    hdmi_in.configure()
    hdmi_in.start()

    if hdmi_out is not None:
        print("Configuring HDMI_OUT to match HDMI_IN mode...")
        hdmi_out.configure(hdmi_in.mode)
        hdmi_out.start()

    # 
    print(f"Benchmarking compute time for {TEST_FRAMES} frames...")
    times_ms = []

    for i in range(TEST_FRAMES):
        frame = hdmi_in.readframe()
        img = np.array(frame)
        if img.ndim == 3 and img.shape[2] >= 3:
            bgr = img[:, :, :3]
        else:
            raise RuntimeError(f"Unexpected frame shape: {img.shape}")
        t0 = time.perf_counter()
        out = cartoon_filter(bgr, EDGE_THRESH, BRIGHT_FACTOR, QUANT_LEVELS, THICKEN)
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

        # Optional: show output on HDMI_OUT (NOT included in timing)
        if hdmi_out is not None:
            outframe = hdmi_out.newframe()
            # outframe might be 4-channel; write into first 3 and preserve alpha
            if outframe.ndim == 3 and outframe.shape[2] >= 3:
                outframe[:, :, :3] = out
            hdmi_out.writeframe(outframe)

        # progress indicator
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{TEST_FRAMES} frames done")

    # ---- Results ----
    t = np.array(times_ms, dtype=np.float64)
    mean_ms = float(t.mean())
    p95_ms = float(np.percentile(t, 95))
    max_ms = float(t.max())
    fps_equiv = 1000.0 / mean_ms if mean_ms > 0 else 0.0

    print("\n=== Python Algorithm Performance (HDMI_IN frames) ===")
    print(f"Frames tested: {len(t)}")
    print(f"Mean: {mean_ms:.2f} ms/frame")
    print(f"P95 : {p95_ms:.2f} ms/frame")
    print(f"Max : {max_ms:.2f} ms/frame")
    print(f"FPS equivalent (mean): {fps_equiv:.2f}")
    print(f"Meets 60fps budget (16.67 ms/frame)? {'YES' if mean_ms <= 16.67 else 'NO'}")

    print("\nStopping HDMI...")
    try:
        hdmi_in.stop()
    except Exception:
        pass
    if hdmi_out is not None:
        try:
            hdmi_out.stop()
        except Exception:
            pass

    print("Done.")

In [5]:
tracemalloc.start()
main()
current, peak = tracemalloc.get_traced_memory()
print(f"Peak   : {peak / (1024**2):.1f} MB")
tracemalloc.stop()

Configuring HDMI_IN...
Benchmarking compute time for 200 frames...
  50/200 frames done
  100/200 frames done
  150/200 frames done
  200/200 frames done

=== Python Compute Performance (HDMI_IN frames, compute-only timing) ===
Frames tested: 200
Mean: 1831.29 ms/frame
P95 : 1939.03 ms/frame
Max : 1965.25 ms/frame
FPS equivalent (mean): 0.55
Meets 60fps budget (16.67 ms/frame)? NO

Stopping HDMI...
Done.
Peak   : 63.4 MB
